Link to guide: https://pennylane.ai/demos/tutorial_qubit_rotation

We start by importing the neccesary libraries

In [1]:
import pennylane as qp
from jax import numpy as np
import jax

Next, we need to define a quantum device. According to PennyLane the definition is: Any computational object that can apply quantum operations and return a measurement value is called a quantum device.

In [2]:
dev1 = qp.device("lightning.qubit", wires=1) # Use quantum device based on the qubit model

Next we construct a quantum node (or QNode). According to PennyLane the definition is: QNodes are an abstract encapsulation of a quantum function, described by a quantum circuit. QNodes are bound to a particular quantum device, which is used to evaluate expectation and variance values of this circuit.

In [4]:
@qp.qnode(dev1)
def circuit(params):
    qp.RX(params[0], wires=0)
    qp.RY(params[1], wires=0)
    return qp.expval(qp.PauliZ(0))

To evaluate the quantum node, we simply call the function with some appropriate numerical inputs.

In [5]:
params = np.array([0.54, 0.12])
print(circuit(params))

0.85154057


Next we can calculate the gradient of the quantum node. PennyLane incorporates both analytic differentiation, as well as numerical methods. 

In [6]:
dcircuit = jax.grad(circuit, argnums=0)

The returned function represents the derivative of the QNode with respect to the argument specified in `argnums`. Because the argument has two elements, the returned gradient is two-dimensional. We can evaluate the gradient function at any point in the parameter space. An example is:

In [7]:
print(dcircuit(params))

[-0.5104387  -0.10267819]


PennyLane QNodes also support using multiple positional arguemnts and keyword arguments instead. The about function could also be specified in the following way:

In [8]:
@qp.qnode(dev1)
def circuit2(phi1, phi2):
    qp.RX(phi1, wires=0)
    qp.RY(phi2, wires=0)
    return qp.expval(qp.PauliZ(0))

If we use multiple positional arguments, then we need to change slightly how to get the gradient of the circuit. For an example see the following:

In [9]:
phi1 = np.array(0.54)
phi2 = np.array(0.12)

dcircuit = jax.grad(circuit2, argnums=[0, 1])
print(dcircuit(phi1, phi2))

(Array(-0.5104387, dtype=float32), Array(-0.10267819, dtype=float32))


PennyLane offers a powerful and flexible interface for gradient-based optimization. When using the JAX interface, we can leverage any JAX-compatible optimizer, such as those provided by Optax or JAXopt, to optimize our hybrid quantum-classical cost functions.

We want to use a JAX-compatible otimizer to optimize thw two circuit parameters $\phi_1$ and $\phi_2$ such taht the qubit, originally in state $\ket{0}$, is rotated to be in state $\ket{1}$. 

This is equivalent to measuring a Pauli-Z expectation value of -1, since the state $\ket{1}$ is an eigenvector of the Pauli-Z matrix with eigenvalue $\lambda=-1$.

To do this, we apply the fundamental techniques known from classical machine learning. First we defined a cost function:

In [10]:
def cost(x):
    return circuit(x)

Next choose some initial values for $\phi_1$ and $\phi_2$:

In [11]:
init_params = np.array([0.011, 0.012])
print(cost(init_params))

0.9998675


Finally, we can use an optimizer to update the circuit parameters for 100 steps. We use gradient descent

In [12]:
import jaxopt

# initialise the optimizer
opt = jaxopt.GradientDescent(cost, stepsize=0.4, acceleration = False)

# set the number of steps
steps = 100
# set the initial parameter values
params = init_params
opt_state = opt.init_state(params)

for i in range(steps):
    # update the circuit parameters
    params, opt_state = opt.update(params, opt_state)

    if (i + 1) % 5 == 0:
        print("Cost after step {:5d}: {: .7f}".format(i + 1, cost(params)))

print("Optimized rotation angles: {}".format(params))

Cost after step     5:  0.9961779
Cost after step    10:  0.8974943
Cost after step    15:  0.1440490
Cost after step    20: -0.1536721
Cost after step    25: -0.9152496
Cost after step    30: -0.9994046
Cost after step    35: -0.9999964
Cost after step    40: -1.0000000
Cost after step    45: -1.0000000
Cost after step    50: -1.0000000
Cost after step    55: -1.0000000
Cost after step    60: -1.0000000
Cost after step    65: -1.0000000
Cost after step    70: -1.0000000
Cost after step    75: -1.0000000
Cost after step    80: -1.0000000
Cost after step    85: -1.0000000
Cost after step    90: -1.0000000
Cost after step    95: -1.0000000
Cost after step   100: -1.0000000
Optimized rotation angles: [7.1526556e-18 3.1415925e+00]
